# Meta-analysis across phenotypes — 1kg_eur

Pools each estimator across phenotypes and asks whether the phenotypes agree.
Reads `09_estimators.ipynb`'s `estimators_<run>.tsv` and computes nothing new
about any single phenotype — every `est`/`se` here came from
`he_estimators.jackknife`.

**Two senses of "meta-analysis", and this notebook does both:**

1. **Inverse-variance pooling.** Fixed-effect and DerSimonian–Laird
   random-effects pooled estimates per `(estimator, class set)`, overall and per
   trait category, with Cochran's *Q*, *I²* and *τ²*. Fixed-effect asks "if
   every phenotype shared one value, what is it"; random-effects allows a real
   spread and widens the interval accordingly. When `I²` is large the
   fixed-effect interval is not believable and the random-effects one is the
   number to quote.
2. **The cross-phenotype comparison plots** from
   `archive/04_process_shards/notebooks/remote/eur_D2_meta_analysis_cells.md`:
   heritability across the covariate staircase, sibling excess against shared
   environment, FS excess against PO excess, and each estimator against
   `h2_Unrel`.

**A caveat that applies to all of it.** These phenotypes are measured on
overlapping sets of the same participants, so their estimates are *not*
independent draws. Pooling treats them as if they were, which understates the
pooled SE and inflates apparent heterogeneity. The pooled numbers are a summary
of this cohort's estimates, not an inference about a population of traits —
quote them as such, and lean on the per-phenotype values in
`09_estimators.ipynb` for anything load-bearing.

Jackknife blocks are random over individuals and split families, so SEs of the
related-range estimators are somewhat optimistic to begin with.

**Prerequisites:** `09_estimators.ipynb`.

## Setup

In [ ]:
import os
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
EST_DIR = (f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
           f"/07_estimators/he_regression")
OUT_DIR = (f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
           f"/07_estimators/meta_analysis")
os.makedirs(f"{OUT_DIR}/plots", exist_ok=True)

EST_PATH = f"{EST_DIR}/estimators_{RUN_DIR}.tsv"
assert os.path.isfile(EST_PATH), (
    f"missing {EST_PATH} — run 09_estimators.ipynb first"
)
E = pd.read_csv(EST_PATH, sep="\t")

# ── shared style (matches the other notebooks) ────────────────────────────────
FIG_TRANSPARENT = False   # white background; True for transparent
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#ffffff", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
# The dark end stops at #215596 rather than #104281: the darkest step was
# close enough to the black reference line that a point sitting on the line
# was hard to pick out. Still passes the ordinal checks -- monotone
# lightness, adjacent dL >= 0.06, light end clear of the surface.
RAMP = ["#86b6ef", "#4b93e8", "#2f76cf", "#215596"]     # covariate sets are ordered
CLASS_COL = {"noPO": "#eb6834", "pooled": "#2a78d6", "PO": "#1baf7a"}
FONT_STACK = ["Helvetica Neue", "Helvetica", "Liberation Sans", "Nimbus Sans",
              "Arial", "DejaVu Sans"]
from matplotlib import font_manager as _fm
_have = {x.name for x in _fm.fontManager.ttflist}
FONT = next((f for f in FONT_STACK if f in _have), "DejaVu Sans")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "savefig.transparent": FIG_TRANSPARENT,
    "figure.dpi": 120, "font.family": FONT, "font.size": 9,
    "mathtext.fontset": "custom", "mathtext.rm": FONT,
    "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold",
    "axes.edgecolor": INK2, "axes.linewidth": 0.7, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.spines.top": True, "axes.spines.right": True,
    "axes.grid": True, "axes.grid.which": "major", "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.5, "grid.alpha": 0.7,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "legend.frameon": False,
})

TRANSFORMS = sorted(E["transform"].unique())
COVSETS_ALL = sorted(E["covset"].unique())
COVSETS = [c for c in ("base", "base_pcs", "base_pcs_zip3", "base_pcs_zip3_ses")
           if c in COVSETS_ALL] or COVSETS_ALL
REF_TF = "invnorm" if "invnorm" in TRANSFORMS else TRANSFORMS[0]
REF_CS = "base_pcs" if "base_pcs" in COVSETS else COVSETS[-1]

print(f"{len(E):,} estimator rows | {E['phenotype'].nunique()} phenotypes")
print(f"transforms: {TRANSFORMS}\ncovsets:    {COVSETS}")
print(f"reference:  {REF_TF} / {REF_CS}")
print(f"font:       {FONT}")
print(f"out:        {OUT_DIR}")

## Trait categories

Same grouping as the archived eur_D2 meta cells, so the two runs' figures are
comparable. A phenotype not listed falls into `uncategorised` rather than being
dropped — silently losing a trait from a pooled estimate is worse than an ugly
category label.

In [ ]:
CATEGORY = {
    **dict.fromkeys(["height", "weight", "bmi", "waist_circumference",
                     "hip_circumference", "waist_hip_ratio"], "anthropometric"),
    **dict.fromkeys(["systolic_bp", "diastolic_bp", "heart_rate"], "cardiovascular"),
    **dict.fromkeys(["glucose", "hba1c", "hdl_cholesterol", "ldl_cholesterol",
                     "triglycerides", "total_cholesterol"], "metabolic"),
    **dict.fromkeys(["hemoglobin", "wbc_count", "platelet_count", "rbc_count",
                     "hematocrit", "mch", "mcv", "mchc", "neutrophil_count",
                     "lymphocyte_count", "monocyte_count", "basophil_count",
                     "eosinophil_count"], "blood cells"),
    **dict.fromkeys(["creatinine", "alt"], "liver & kidney"),
    **dict.fromkeys(["alcohol_audit_c_score", "cigarettes_per_day", "pss_score",
                     "social_support_score", "loneliness_score", "eds_score"],
                    "behavioural"),
}
CAT_ORDER = ["anthropometric", "cardiovascular", "metabolic", "blood cells",
             "liver & kidney", "behavioural", "uncategorised"]

E["category"] = E["phenotype"].map(CATEGORY).fillna("uncategorised")
present = [c for c in CAT_ORDER if c in set(E["category"])]
print(E.groupby("category")["phenotype"].nunique().reindex(present).to_string())
unc = sorted(E.loc[E["category"] == "uncategorised", "phenotype"].unique())
if unc:
    print(f"\nuncategorised ({len(unc)}): {unc}")

## Pooling

`pool()` takes one group of per-phenotype estimates and returns both pooled
values with their heterogeneity statistics:

- weights `w = 1/se²`; fixed-effect `θ_F = Σwθ / Σw`, `SE = 1/√Σw`
- Cochran's `Q = Σw(θ − θ_F)²` on `k − 1` df; `I² = max(0, (Q − (k−1))/Q)`
- DerSimonian–Laird `τ² = max(0, (Q − (k−1)) / (Σw − Σw²/Σw))`
- random-effects re-weights with `w* = 1/(se² + τ²)`

Rows with a missing or non-positive SE are dropped — the jackknife returns NaN
unless every block produced a finite estimate, and a zero SE would take infinite
weight. `k` is reported so a pooled value resting on two phenotypes is visible
as such.

In [ ]:
def pool(est, se):
    est = np.asarray(est, float); se = np.asarray(se, float)
    ok = np.isfinite(est) & np.isfinite(se) & (se > 0)
    est, se = est[ok], se[ok]
    k = len(est)
    if k == 0:
        return dict(k=0, fe=np.nan, fe_se=np.nan, re=np.nan, re_se=np.nan,
                    Q=np.nan, I2=np.nan, tau2=np.nan)
    w = 1.0 / se ** 2
    fe = float((w * est).sum() / w.sum())
    fe_se = float(np.sqrt(1.0 / w.sum()))
    if k == 1:
        return dict(k=1, fe=fe, fe_se=fe_se, re=fe, re_se=fe_se,
                    Q=np.nan, I2=np.nan, tau2=0.0)
    Q = float((w * (est - fe) ** 2).sum())
    df = k - 1
    I2 = max(0.0, (Q - df) / Q) if Q > 0 else 0.0
    denom = w.sum() - (w ** 2).sum() / w.sum()
    tau2 = max(0.0, (Q - df) / denom) if denom > 0 else 0.0
    ws = 1.0 / (se ** 2 + tau2)
    re = float((ws * est).sum() / ws.sum())
    re_se = float(np.sqrt(1.0 / ws.sum()))
    return dict(k=k, fe=fe, fe_se=fe_se, re=re, re_se=re_se,
                Q=Q, I2=I2, tau2=tau2)


def pool_table(df, by):
    rows = []
    for key, g in df.groupby(by, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(by, key)), **pool(g["est"], g["se"])})
    return pd.DataFrame(rows)


# n_pairs is bookkeeping, not an estimate -- it has no SE and pooling a count
# produces a meaningless "pooled" pair count. Kept aside for reporting.
BOOKKEEPING = {"n_pairs"}
PAIR_COUNTS = E[E["estimator"].isin(BOOKKEEPING)]
EST = E[~E["estimator"].isin(BOOKKEEPING)]

meta = pool_table(EST, ["estimator", "classes", "transform", "covset"])
meta_cat = pool_table(EST, ["estimator", "classes", "transform", "covset", "category"])

meta.to_csv(f"{OUT_DIR}/meta_pooled_{RUN_DIR}.tsv", sep="\t", index=False)
meta_cat.to_csv(f"{OUT_DIR}/meta_pooled_by_category_{RUN_DIR}.tsv", sep="\t", index=False)
print(f"{len(meta):,} pooled rows, {len(meta_cat):,} by category -> {OUT_DIR}")

print("\npairs entering the first-degree fits (first combo):")
print(PAIR_COUNTS.groupby("classes")["est"].first().round(0).to_string())

ref = meta[(meta["transform"] == REF_TF) & (meta["covset"] == REF_CS)]
print(f"\n{REF_TF} / {REF_CS}, pooled over {E['phenotype'].nunique()} phenotypes:")
print(ref[["estimator", "classes", "k", "fe", "fe_se", "re", "re_se", "I2", "tau2"]]
      .sort_values("estimator").round(4).to_string(index=False))

## Figure 1 — pooled estimates, fixed vs random effects

One row per `(estimator, class set)`. Where the two markers separate, the
phenotypes disagree more than their own SEs allow and the random-effects
interval is the honest one. `I²` is printed beside each row.

In [ ]:
LADDER = ["h2_Unrel", "h2_FS", "h2_PedW25", "h2_OneSlope", "h2_Rel",
          "h2_OneSlopeOffsets", "h2_RelOffsets", "b2_FS", "b2_step"]
d = ref[ref["estimator"].isin(LADDER) & (ref["k"] > 0)].copy()
d["order"] = d["estimator"].map({e: i for i, e in enumerate(LADDER)})
d = d.sort_values(["order", "classes"], ascending=[False, True]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.2, 0.34 * len(d) + 1.4), constrained_layout=True)
y = np.arange(len(d))
ax.errorbar(d["fe"], y - 0.13, xerr=d["fe_se"], fmt="o", ms=4, lw=0.8,
            color=INK, alpha=0.85, label="fixed effect")
ax.errorbar(d["re"], y + 0.13, xerr=d["re_se"], fmt="s", ms=4, lw=0.8,
            color="#eb6834", alpha=0.85, label="random effects")
ax.axvline(0, color=INK2, lw=0.8)
ax.set_yticks(y)
ax.set_yticklabels([f"{r.estimator}  [{r.classes}]" for r in d.itertuples()], fontsize=7.5)
for i, r in enumerate(d.itertuples()):
    if np.isfinite(r.I2):
        ax.annotate(f"I²={100*r.I2:.0f}%  k={r.k}", (1.0, i), xycoords=("axes fraction", "data"),
                    ha="right", va="center", fontsize=6.5, color=MUTED)
ax.set_xlabel("pooled estimate ±1 SE")
ax.set_title(f"Pooled across {E['phenotype'].nunique()} phenotypes — {REF_TF}, {REF_CS}",
             loc="left")
ax.legend(fontsize=7.5, loc="lower right")
png = f"{OUT_DIR}/plots/pooled_fe_vs_re.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")

## Figure 2 — unrelated-pair heritability across trait categories

Pooled `h2_Unrel` per category, random effects, with `k` and `I²` beside each
row and the all-trait pooled value as a reference line. This is the
heritability from unrelated pairs only, so it is the SNP-based baseline every
other estimator is compared against.

Note what this is *not*: `h2_Unrel` is fitted on `a_ij < 0.02`, where no pair
is classified, so it does not vary across the `noPO`/`pooled`/`PO` class sets —
`09_estimators.ipynb` asserts `other` and `pooled` agree there to within 1e-9,
and `PO` has no bins that low at all. Splitting it by pair class would give
identical values by construction; splitting it by trait is the comparison with
something in it.

In [ ]:
cats = (meta_cat[(meta_cat["estimator"] == "h2_Unrel") & (meta_cat["classes"] == "noPO")
                 & (meta_cat["transform"] == REF_TF) & (meta_cat["covset"] == REF_CS)
                 & (meta_cat["k"] > 0)]
        .copy())
cats["order"] = cats["category"].map({c: i for i, c in enumerate(CAT_ORDER)})
cats = cats.sort_values("order", ascending=False).reset_index(drop=True)

overall = ref[(ref["estimator"] == "h2_Unrel") & (ref["classes"] == "noPO")]
fig, ax = plt.subplots(figsize=(6.6, 0.42 * len(cats) + 1.5), constrained_layout=True)
y = np.arange(len(cats))
if len(overall):
    o = overall.iloc[0]
    ax.axvspan(o["re"] - o["re_se"], o["re"] + o["re_se"], color=GRID, alpha=0.8,
               lw=0, zorder=0)
    ax.axvline(o["re"], color=INK2, lw=0.9, ls=(0, (4, 2)), zorder=1,
               label=f"all traits: {o['re']:.3f} ± {o['re_se']:.3f}  (k={int(o['k'])})")
ax.errorbar(cats["re"], y, xerr=cats["re_se"], fmt="o", ms=5, lw=1,
            color=CLASS_COL["noPO"], zorder=3)
ax.set_yticks(y); ax.set_yticklabels(cats["category"], fontsize=8)
for i_, r in enumerate(cats.itertuples()):
    lab = f"k={r.k}" + (f"  I²={100*r.I2:.0f}%" if np.isfinite(r.I2) else "")
    ax.annotate(lab, (1.0, i_), xycoords=("axes fraction", "data"), ha="right",
                va="center", fontsize=6.5, color=MUTED)
ax.set_xlabel("pooled $h^2_{\\mathrm{Unrel}}$ (random effects) ±1 SE")
ax.set_title(f"Unrelated-pair heritability by trait category — {REF_TF}, {REF_CS}",
             loc="left")
ax.legend(fontsize=7.5, loc="lower right")
png = f"{OUT_DIR}/plots/h2_unrel_by_category.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")
print(cats[["category", "k", "re", "re_se", "I2", "tau2"]].round(4).to_string(index=False))

## Figure 3 — heritability across the covariate staircase

Pooled `h2_Unrel` as covariates are added: age/sex, then PCs, then 3-digit zip,
then SES. A slope downward means the covariates were absorbing something the
earlier model was attributing to relatedness. Flat means they were not.

In [ ]:
stair = meta[(meta["estimator"] == "h2_Unrel") & (meta["classes"] == "noPO")
             & (meta["covset"].isin(COVSETS))]
fig, ax = plt.subplots(figsize=(6.2, 3.8), constrained_layout=True)
for i, tf in enumerate(TRANSFORMS):
    s = stair[stair["transform"] == tf].set_index("covset").reindex(COVSETS)
    x = np.arange(len(COVSETS))
    ax.errorbar(x, s["re"], yerr=s["re_se"], marker="o", ms=4.5, lw=1.1,
                color=RAMP[i % len(RAMP)], label=tf, capsize=2)
ax.set_xticks(np.arange(len(COVSETS)))
ax.set_xticklabels(COVSETS, rotation=20, ha="right", fontsize=7.5)
ax.set_ylabel("pooled $h^2_{\\mathrm{Unrel}}$ (random effects)")
ax.set_title("Heritability across the covariate staircase", loc="left")
ax.legend(title="transform", fontsize=7.5, title_fontsize=8)
png = f"{OUT_DIR}/plots/h2_covariate_staircase.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")

## Figure 4 — sibling excess against shared environment, and FS against PO

Left: per phenotype, `excess` in the FS class against `b2_FS` on `noPO`. Excess
is the pair-weighted mean cross-product above the additive prediction; `b2_FS`
is the shared-environment term read off the intercept. They should agree in
sign — both are measuring the elevation of the first-degree bins — and the
scatter is how much they disagree in magnitude.

Right: FS excess against PO excess. PO and FS pairs sit at the same `a_ij` but
share neither dominance nor a rearing environment, so points above the diagonal
are where the classification bought something.

In [ ]:
def per_pheno(estimator, classes):
    return (E[(E["estimator"] == estimator) & (E["classes"] == classes)
              & (E["transform"] == REF_TF) & (E["covset"] == REF_CS)]
            .set_index("phenotype")["est"])

ex_fs, ex_po = per_pheno("excess", "FS"), per_pheno("excess", "PO")
b2 = per_pheno("b2_FS", "noPO")
cat = E.drop_duplicates("phenotype").set_index("phenotype")["category"]
CAT_COL = dict(zip(CAT_ORDER, ["#2a78d6", "#e34948", "#008300", "#4a3aa7",
                               "#8c564b", "#eb6834", "#898781"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 4.2), constrained_layout=True)
for axis, (xs, ys, xl, yl, diag) in zip(
        (ax1, ax2),
        [(b2, ex_fs, "$b^2_{FS}$  (noPO)", "FS excess", False),
         (ex_po, ex_fs, "PO excess", "FS excess", True)]):
    idx = xs.index.intersection(ys.index)
    for c in [c for c in CAT_ORDER if c in set(cat.reindex(idx))]:
        sel = [p for p in idx if cat.get(p) == c]
        axis.scatter(xs[sel], ys[sel], s=42, color=CAT_COL[c], edgecolor="none",
                     label=c, zorder=3)
    for p in idx:
        axis.annotate(p, (xs[p], ys[p]), fontsize=6, color=MUTED,
                      xytext=(3, 2), textcoords="offset points")
    axis.axhline(0, color=INK2, lw=0.6); axis.axvline(0, color=INK2, lw=0.6)
    if diag:
        lim = np.nanmax(np.abs(np.r_[xs[idx].values, ys[idx].values])) * 1.15
        axis.plot([-lim, lim], [-lim, lim], color=MUTED, lw=0.8, ls="--", zorder=1)
    axis.set_xlabel(xl); axis.set_ylabel(yl)
ax1.set_title("excess vs shared environment", loc="left")
ax2.set_title("FS vs PO excess", loc="left")
ax1.legend(fontsize=7, loc="best")
png = f"{OUT_DIR}/plots/excess_vs_b2_and_fs_vs_po.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")

## Figure 5 — each estimator against `h2_Unrel`

`h2_Unrel` is the SNP-based baseline from unrelated pairs. Every other
estimator adds related pairs, and the vertical gap from the diagonal is what
those pairs contribute — pedigree effects, shared environment, dominance, or
assortative mating, depending on the estimator. A point on the diagonal means
the related pairs said nothing the unrelated ones had not.

In [ ]:
AGAINST = [e for e in ["h2_FS", "h2_PedW25", "h2_Rel", "h2_RelOffsets"]
           if e in set(E["estimator"])]
base = per_pheno("h2_Unrel", "noPO")

ncol = min(3, len(AGAINST))
nrow = int(np.ceil(len(AGAINST) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow),
                         squeeze=False, constrained_layout=True)
for i, est in enumerate(AGAINST):
    ax = axes[i // ncol][i % ncol]
    ys = per_pheno(est, "noPO")
    idx = base.index.intersection(ys.index)
    if not len(idx):
        ax.set_visible(False); continue
    ax.scatter(base[idx], ys[idx], s=34, color=CLASS_COL["noPO"], edgecolor="none",
               zorder=3)
    lo = float(np.nanmin(np.r_[base[idx].values, ys[idx].values]))
    hi = float(np.nanmax(np.r_[base[idx].values, ys[idx].values]))
    pad = 0.08 * (hi - lo or 1)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=MUTED, lw=0.8, ls="--")
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
    ax.set_title(est, fontsize=9)
    ax.set_xlabel("$h^2_{\\mathrm{Unrel}}$"); ax.set_ylabel(est)
for j in range(len(AGAINST), nrow * ncol):
    axes[j // ncol][j % ncol].set_visible(False)
fig.suptitle(f"Estimators against the unrelated-pair baseline — {REF_TF}, {REF_CS}",
             fontsize=10)
png = f"{OUT_DIR}/plots/estimator_vs_h2_unrel.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")

## Heterogeneity

Where `I²` is high the phenotypes genuinely disagree and a single pooled number
is the wrong summary. Read this before quoting anything from Figure 1 — and
remember the caveat in the header: these phenotypes share participants, so some
of this apparent heterogeneity is the pooling model being wrong rather than the
traits differing.

In [ ]:
het = (ref[(ref["k"] >= 2) & ref["I2"].notna()]
       .sort_values("I2", ascending=False)
       [["estimator", "classes", "k", "Q", "I2", "tau2", "fe", "fe_se", "re", "re_se"]])
print(f"{REF_TF} / {REF_CS}, most heterogeneous first:")
print(het.round(4).to_string(index=False))

by_cat = (meta_cat[(meta_cat["transform"] == REF_TF) & (meta_cat["covset"] == REF_CS)
                   & (meta_cat["estimator"] == "h2_Unrel")
                   & (meta_cat["classes"] == "noPO") & (meta_cat["k"] > 0)]
          [["category", "k", "re", "re_se", "I2"]])
print("\npooled h2_Unrel by trait category:")
print(by_cat.round(4).to_string(index=False))

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser("~/repos/AOU-covariance/notebooks/12_meta_analysis.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/12_meta_analysis.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")